# Part 1: Variantion on the ERC portfolio<br>
Question 1.e : given volotilities and correlation matrix, compute the ERC portfolio

In [3]:
"""
ERC Portfolio via the Beta Algorithm (Jacobi fixed-point iteration).

Reference setup (Question 1.e):
    - 3 assets, volatilities  sigma = [15%, 20%, 25%]
    - Correlation matrix Rho:
            [[1.00, 0.50, 0.40],
             [0.50, 1.00, 0.30],
             [0.40, 0.30, 1.00]]
    - Goal: find the Equal Risk Contribution (ERC) portfolio weights.

The Beta algorithm
------------------
For a long-only fully-invested portfolio, the ERC condition is
    x_i * beta_i(x) = x_j * beta_j(x)   for all i, j,
which is equivalent to
    x_i  proportional to  1 / beta_i(x).

Because beta depends on x itself, we solve it as a fixed-point iteration
(Jacobi / Picard iteration):

    x_i^{(k+1)} =  beta_i^{-1}(x^{(k)})  /  sum_j beta_j^{-1}(x^{(k)})

If the iteration converges, the limit satisfies x* = F(x*) and is the ERC
portfolio.
"""

import numpy as np


# ---------------------------------------------------------------------------
# 1. Build the covariance matrix from vols and the correlation matrix
# ---------------------------------------------------------------------------
def build_covariance(vols: np.ndarray, corr: np.ndarray) -> np.ndarray:
    """Covariance matrix  Sigma = diag(sigma) @ Rho @ diag(sigma)."""
    D = np.diag(vols)
    return D @ corr @ D


# ---------------------------------------------------------------------------
# 2. Beta of each asset w.r.t. the current portfolio
# ---------------------------------------------------------------------------
def asset_betas(x: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """
    beta_i(x) = (Sigma x)_i / (x' Sigma x)
              = MR_i / sigma_p,
    where MR_i is the marginal risk and sigma_p is the portfolio volatility.
    """
    Sigma_x = Sigma @ x                       # marginal contribution to risk
    portfolio_var = float(x @ Sigma_x)        # x' Sigma x
    return Sigma_x / portfolio_var


# ---------------------------------------------------------------------------
# 3. One step of the Beta algorithm
# ---------------------------------------------------------------------------
def beta_update(x: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """
    Fixed-point update:
        x_i^{new} = (1 / beta_i(x)) / sum_j (1 / beta_j(x))
    The output is always a valid long-only fully-invested portfolio
    (non-negative entries that sum to 1).
    """
    betas = asset_betas(x, Sigma)
    inv_beta = 1.0 / betas                    # inverse beta
    return inv_beta / np.sum(inv_beta)        # normalize to sum = 1


# ---------------------------------------------------------------------------
# 4. Main loop: iterate until convergence
# ---------------------------------------------------------------------------
def solve_erc_beta(
    Sigma: np.ndarray,
    x0: np.ndarray | None = None,
    tol: float = 1e-12,
    max_iter: int = 100000,
) -> tuple[np.ndarray, int, float]:
    """
    Solve for the ERC portfolio using the Beta algorithm.

    Parameters
    ----------
    Sigma      : covariance matrix (n x n), must be positive definite.
    x0         : initial weights (defaults to equal-weight).
    tol        : convergence tolerance on ||x^{(k+1)} - x^{(k)}||_inf.
    max_iter   : safety cap on the number of iterations.

    Returns
    -------
    x_erc      : the converged ERC weights (sum to 1).
    n_iter     : number of iterations actually performed.
    err        : final infinity-norm change between the last two iterates.
    """
    n = Sigma.shape[0]
    if x0 is None:
        x0 = np.ones(n) / n                   # equal-weight start
    x = x0.astype(float).copy()

    err = np.inf
    for k in range(1, max_iter + 1):
        x_new = beta_update(x, Sigma)
        err = np.max(np.abs(x_new - x))
        x = x_new
        if err < tol:
            return x, k, err
    return x, max_iter, err


# ---------------------------------------------------------------------------
# 5. Diagnostic: verify that risk contributions are indeed equal
# ---------------------------------------------------------------------------
def risk_contributions(x: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """
    RC_i = x_i * MR_i = x_i * (Sigma x)_i.
    In an ERC portfolio every RC_i should be identical and they should sum
    to the total portfolio variance.
    """
    return x * (Sigma @ x) / ((x @ Sigma @ x)**0.5)


# ---------------------------------------------------------------------------
# 6. Run the example from Question 1.e
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # Asset volatilities
    vols = np.array([0.15, 0.20, 0.25])

    # Correlation matrix
    corr = np.array(
        [
            [1.00, 0.50, 0.40],
            [0.50, 1.00, 0.30],
            [0.40, 0.30, 1.00],
        ]
    )

    # Build Sigma
    Sigma = build_covariance(vols, corr)

    print("Covariance matrix Sigma:")
    print(np.round(Sigma, 6))
    print()

    # Solve for the ERC weights
    x_erc, n_iter, err = solve_erc_beta(Sigma)

    print(f"Converged in {n_iter} iterations, final step error = {err:.2e}")
    print()
    print("ERC portfolio weights:")
    for i, w in enumerate(x_erc, start=1):
        print(f"  x_{i} = {w:.6f}")
    print(f"  sum  = {x_erc.sum():.6f}")
    print()

    # Portfolio volatility
    sigma_p = float(np.sqrt(x_erc @ Sigma @ x_erc))
    print(f"Portfolio volatility = {sigma_p:.6f}  ({sigma_p*100:.4f}%)")
    print()

    # Risk contributions (should all be equal)
    rc = risk_contributions(x_erc, Sigma)
    print("Risk contributions  RC_i = x_i * (Sigma x)_i:")
    for i, val in enumerate(rc, start=1):
        print(f"  RC_{i} = {val:.8f}")
    print(f"  max - min = {rc.max() - rc.min():.2e}  (should be ~0 for ERC)")
    print()

    # Relative risk contributions (share of total variance)
    rc_share = rc / rc.sum()
    print("Relative risk contributions (share of total variance):")
    for i, val in enumerate(rc_share, start=1):
        print(f"  RC_{i} / sum = {val:.6f}")

Covariance matrix Sigma:
[[0.0225 0.015  0.015 ]
 [0.015  0.04   0.015 ]
 [0.015  0.015  0.0625]]

Converged in 30 iterations, final step error = 6.12e-13

ERC portfolio weights:
  x_1 = 0.410400
  x_2 = 0.321907
  x_3 = 0.267693
  sum  = 1.000000

Portfolio volatility = 0.149190  (14.9190%)

Risk contributions  RC_i = x_i * (Sigma x)_i:
  RC_1 = 0.04972998
  RC_2 = 0.04972998
  RC_3 = 0.04972998
  max - min = 7.66e-14  (should be ~0 for ERC)

Relative risk contributions (share of total variance):
  RC_1 / sum = 0.333333
  RC_2 / sum = 0.333333
  RC_3 / sum = 0.333333


Question 2